# SR-DAS — manuscript evidence pack (notebook 4)

**Self-contained**: downloads its own data and contains all code. It produces every table and figure the manuscript needs, using the **frozen specification of notebook 3** (the fingerprint printed below must equal the one from notebook 3).

| Section | Output |
|---|---|
| 3 | **Table 1** descriptive statistics; **Figure 2** correlations rise in stress |
| 4 | **Table 2** main results (holdout and replication); **Table 3** hypothesis verdicts; **Table 4** Sharpe ratios with 95% CIs; **Table 5** crisis drawdowns; **Figures 1, 3–7** |
| 5 | **Table 6** robustness of B-SR-DAS (costs 0/10/25/50 bps, k = 5/15, min-variance weights, 12 industries); **Figure 8** |
| 6 | **Tables 7–9** S&P 500 replication (point-in-time membership, hypotheses, survivorship check, coverage); **Figures 9–10** |

Everything is saved under `outputs_manuscript/` (tables as CSV + LaTeX + one Excel workbook; figures as 300-dpi PNG + vector PDF) and downloaded as one zip at the end.

**Runtime** on a standard Colab CPU: about 15 minutes without the S&P 500 section, 45–60 minutes with it (the Yahoo download is cached; if notebook 2 saved its cache to Google Drive, point `SP500_DATA_DIR` at it to skip the download).

## 0. Setup

In [ ]:
!pip -q install --upgrade yfinance networkx scikit-learn openpyxl jinja2 > /dev/null

In [ ]:
import os
from IPython.display import display
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120); pd.set_option("display.max_colwidth", 90)

# ---- run options ----
RUN_ROBUSTNESS = True      # Table 6 / Figure 8
RUN_SP500 = True           # Tables 7-9 / Figures 9-10 (slowest part)
N_BOOT = 2000              # bootstrap replications (main tests)
N_BOOT_ROBUST = 1000       # bootstrap replications (robustness grid)
N_RANDOM = 20              # random-k baskets (industries)
N_RANDOM_SP500 = 10        # random-k baskets (stocks)
SAVE_TO_DRIVE = False      # True: mount Google Drive (data cache and outputs persist)
DATA_DIR = "data"
SP500_DATA_DIR = "data"    # e.g. "/content/drive/MyDrive/srdas/sp500/data" to reuse notebook 2's cache
OUT_ROOT = "outputs_manuscript"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/srdas/manuscript/data"
os.makedirs(OUT_ROOT, exist_ok=True)

## 1. Frozen specification (identical to notebook 3)

In [ ]:
import json, hashlib, datetime

# =====================================================================
#  PRE-REGISTERED SPECIFICATION — frozen before the holdout is run.
#  Do NOT edit after the first run; any change must be reported as a deviation.
# =====================================================================
SPEC = {
    "primary_method": "B-SR-DAS (balanced): log-det on robust stress correlation, theta set by balanced_theta (beta=1)",
    "secondary_method": "S-SR-DAS: top 50% by quality, then pure log-det (theta=0) on robust stress correlation",
    "dataset": "Fama-French 49 industry portfolios, daily, value-weighted",
    "holdout_period": ["1970-01-01", "1989-12-31"],
    "replication_period": ["1990-01-01", "latest"],
    "k": 10, "lookback_days": 1260, "rebalance_days": 63, "stress_q": 0.10,
    "sub_len": 504, "sub_step": 126, "agg_q": 0.75, "kernel_a": 0.5, "cost_bps": 10,
    "weights": "equal",
    "hypotheses": {
        "H1": "Primary's out-of-sample stress-day basket correlation < Top-k Quality (paired Wilcoxon, one-sided, alpha=0.05)",
        "H2": "Primary's volatility < EW-All 1/N (block-bootstrap 95% CI of ratio entirely below 1)",
        "H3": "Primary's volatility < pure-diversity SR-DAS theta=0 (95% CI of ratio below 1)",
        "H4": "Primary's Sharpe not significantly below EW-All 1/N (two-sided block bootstrap, alpha=0.05)",
        "H5": "RQ1: stress-day basket correlation < Stress-Cluster rule (one-sided Wilcoxon)",
        "H6": "RQ2: stress-day basket correlation < same method with all-day correlation (one-sided Wilcoxon)",
        "H7": "RQ3: stress-day basket correlation < same method with single-window estimate (one-sided Wilcoxon)",
    },
    "multiple_testing": "Holm correction across all Sharpe comparisons of the primary method",
    "decision_rule": "The holdout verdicts are the headline results; the 1990+ replication was seen during development and is reported as supporting evidence only.",
}
SPEC_JSON = json.dumps(SPEC, indent=2, sort_keys=True)
SPEC_HASH = hashlib.sha256(SPEC_JSON.encode()).hexdigest()[:16]
print(SPEC_JSON)
print("\nSpecification fingerprint:", SPEC_HASH)

EXPECTED = "a1be5ab38d9e20b6"   # fingerprint printed by notebook 3
print("matches notebook 3:", SPEC_HASH == EXPECTED)
assert SPEC_HASH == EXPECTED, "Specification differs from notebook 3 - do not edit the SPEC cell"


## 2. Library
Same code as notebooks 1–3, plus the manuscript helpers (cost re-pricing, Sharpe CIs, figure and table export).

In [ ]:
import math
import time
import warnings
from dataclasses import dataclass, field
from functools import cached_property

import numpy as np
import pandas as pd
import networkx as nx
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=RuntimeWarning)
TRADING_DAYS = 252


In [ ]:
def standardize(X):
    """Column-standardize a T x N array (zero-variance columns left at 0)."""
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)
    sd = np.where(sd > 1e-12, sd, 1.0)
    return (X - mu) / sd


def shrunk_corr(X):
    """Ledoit-Wolf shrunk correlation matrix of a T x N return array.

    Returns are standardized first, so Ledoit-Wolf shrinks the sample
    correlation toward the identity. Works when T < N (as on stress days).
    """
    Z = standardize(np.asarray(X, dtype=float))
    S = LedoitWolf().fit(Z).covariance_
    d = np.sqrt(np.clip(np.diag(S), 1e-12, None))
    C = S / np.outer(d, d)
    np.fill_diagonal(C, 1.0)
    return C


def nearest_psd_corr(C, eps=1e-6):
    """Project a symmetric matrix to a valid (PSD, unit-diagonal) correlation matrix.

    Eigenvalue clipping followed by rescaling to unit diagonal. Rescaling is a
    congruence transform, so positive semi-definiteness is preserved.
    """
    C = (C + C.T) / 2.0
    w, V = np.linalg.eigh(C)
    w = np.clip(w, eps, None)
    C2 = (V * w) @ V.T
    d = np.sqrt(np.diag(C2))
    C2 = C2 / np.outer(d, d)
    C2 = (C2 + C2.T) / 2.0
    np.fill_diagonal(C2, 1.0)
    return C2


def stress_mask(mkt, stress_q):
    """Boolean mask of 'stress days': market return at or below its stress_q quantile."""
    thr = np.quantile(mkt, stress_q)
    return mkt <= thr


def window_starts(T, sub_len, sub_step):
    """Start indices of sub-windows of length sub_len, anchored at the most recent day."""
    if sub_len >= T:
        return [0]
    starts = []
    s = T - sub_len
    while s >= 0:
        starts.append(s)
        s -= sub_step
    return starts[::-1]


def robust_stress_corr(R, mkt, stress_q=0.10, sub_len=504, sub_step=126,
                       agg_q=0.75, min_stress_days=20):
    """Stress-conditioned, temporally robust correlation matrix (the SR-DAS dependence estimate).

    1. Split the look-back into overlapping sub-windows.
    2. In each sub-window keep only stress days (market in its bottom stress_q tail)
       and estimate a Ledoit-Wolf shrunk correlation on those days.
    3. Aggregate pessimistically: for every pair take the agg_q quantile of its
       stress correlation across sub-windows (agg_q > 0.5 = assume the pair is
       as dependent as it has reliably been in bad times).
    4. Project to the nearest valid correlation matrix so log-det is well defined.

    R: T x N array of returns, mkt: length-T array of market returns.
    Returns (C, n_windows_used).
    """
    R = np.asarray(R, dtype=float)
    mkt = np.asarray(mkt, dtype=float)
    T = R.shape[0]
    mats = []
    for s in window_starts(T, sub_len, sub_step):
        e = min(T, s + sub_len)
        m = stress_mask(mkt[s:e], stress_q)
        if m.sum() < min_stress_days:
            continue
        mats.append(shrunk_corr(R[s:e][m]))
    if not mats:  # fall back to all-day correlation
        return shrunk_corr(R), 0
    stack = np.stack(mats)
    C = np.quantile(stack, agg_q, axis=0) if len(mats) > 1 else stack[0]
    return nearest_psd_corr(C), len(mats)


def lower_tail_dependence(R, u=0.10):
    """Empirical (non-parametric) lower tail-dependence coefficients.

    lambda_ij = P(U_i <= u, U_j <= u) / u, with U the column ranks scaled to (0,1).
    This is the copula-free estimator used in tail-dependence clustering work.
    """
    R = np.asarray(R, dtype=float)
    T = R.shape[0]
    U = (pd.DataFrame(R).rank(axis=0).values) / (T + 1.0)
    I = (U <= u).astype(float)
    lam = (I.T @ I) / T / u
    lam = np.clip(lam, 0.0, 1.0)
    np.fill_diagonal(lam, 1.0)
    return lam


In [ ]:
def rank_normal(x):
    """Map a cross-section to standard-normal scores via ranks (robust to outliers)."""
    x = pd.Series(x)
    r = x.rank(method="average")
    return pd.Series(norm.ppf((r - 0.5) / len(x)), index=x.index)


def quality_scores(R_df, vol_window=252, mom_window=252, mom_skip=21):
    """Composite quality score z_i from price-only signals with out-of-sample evidence.

    - Low volatility: minus the standard deviation of the last vol_window daily returns.
    - 12-1 momentum: return from t-mom_window to t-mom_skip (skips the last month).
    Each signal is rank-normalized, averaged and re-normalized to a z-score.
    Uses only data up to the rebalance date (no look-ahead).
    """
    X = R_df.values
    vol = X[-vol_window:].std(axis=0, ddof=1)
    seg = X[-mom_window:-mom_skip] if mom_skip > 0 else X[-mom_window:]
    mom = np.prod(1.0 + seg, axis=0) - 1.0
    z = (rank_normal(-vol).values + rank_normal(mom).values) / 2.0
    z = (z - z.mean()) / (z.std(ddof=1) + 1e-12)
    return pd.Series(z, index=R_df.columns)


In [ ]:
def affine_corr_kernel(C, a=0.5):
    """Similarity kernel K = (1-a) C + a 11^T.

    Maps correlation -1..1 to similarity 0..1 for a=0.5, so that NEGATIVE
    correlation counts as MORE diverse (a plain correlation kernel would
    penalize hedging pairs, since det([[1,r],[r,1]]) = 1-r^2).
    K is PSD because it is a sum of PSD matrices. One can show
    det(K_S) = (1-a)^k det(C_S) (1 + a/(1-a) * 1' C_S^{-1} 1),
    i.e. it rewards both low generalized correlation and a low-risk
    minimum-variance combination of the chosen assets.
    """
    return (1.0 - a) * C + a * np.ones_like(C)


def greedy_logdet(L, k):
    """Greedy MAP for log det(L_S) with |S| = k (fast incremental Cholesky).

    log det is submodular, so greedy has the classic (1 - 1/e) guarantee for
    the monotone case and is the standard DPP MAP heuristic otherwise.
    Returns the list of selected indices (in order of selection).
    """
    N = L.shape[0]
    k = min(k, N)
    cis = np.zeros((k, N))
    di2 = np.diag(L).astype(float).copy()
    sel = []
    for j in range(k):
        score = di2.copy()
        score[sel] = -np.inf
        i = int(np.argmax(score))
        if di2[i] <= 1e-12:
            break
        sel.append(i)
        ei = (L[i, :] - cis[:j, i] @ cis[:j, :]) / np.sqrt(di2[i])
        cis[j, :] = ei
        di2 = di2 - ei ** 2
    return sel


def logdet_select(C, z, k, theta=0.5, a=0.5):
    """SR-DAS selection rule on a given correlation estimate C and quality z.

    L = diag(q) K diag(q) with q = exp(theta * z), so that
    log det L_S = 2 theta sum_{i in S} z_i + log det K_S
    (theta trades quality against diversity; theta=0 is pure diversity).
    """
    K = affine_corr_kernel(C, a)
    q = np.exp(theta * np.asarray(z))
    L = K * np.outer(q, q)
    return greedy_logdet(L, k)


def cluster_pick(D, z, k, method="average"):
    """Cluster-then-pick baseline: hierarchical clustering on distance D into k
    clusters, then the highest-quality asset from each cluster."""
    D = np.asarray(D, dtype=float)
    D = (D + D.T) / 2.0
    np.fill_diagonal(D, 0.0)
    D = np.clip(D, 0.0, None)
    Z = linkage(squareform(D, checks=False), method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    z = np.asarray(z)
    sel = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        sel.append(int(idx[np.argmax(z[idx])]))
    # fcluster can return fewer than k clusters on ties; top up by quality
    if len(sel) < k:
        rest = [i for i in np.argsort(-z) if i not in sel]
        sel += rest[: k - len(sel)]
    return sel[:k]


def corr_distance(C):
    return np.sqrt(np.clip(2.0 * (1.0 - C), 0.0, None))


def mst_periphery_select(C, k):
    """Pozzi, Di Matteo & Aste (2013)-style selection: build the Minimum Spanning
    Tree on correlation distance and pick the k most peripheral nodes.

    Hybrid centrality = rank(degree) + rank(betweenness) + rank(closeness)
    + rank(-eccentricity); lowest values are the most peripheral.
    (Simplified version of their hybrid measure, on the MST only.)
    """
    from scipy.sparse.csgraph import minimum_spanning_tree
    N = C.shape[0]
    D = corr_distance(C) + 1e-9          # tiny offset: zero distances would be dropped as non-edges
    np.fill_diagonal(D, 0.0)
    mst = minimum_spanning_tree(D).tocoo()
    T = nx.Graph()
    T.add_nodes_from(range(N))
    T.add_weighted_edges_from(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    deg = pd.Series(dict(T.degree(weight=None)))
    btw = pd.Series(nx.betweenness_centrality(T, weight="weight"))
    clo = pd.Series(nx.closeness_centrality(T, distance="weight"))
    ecc = pd.Series(nx.eccentricity(T))
    score = deg.rank() + btw.rank() + clo.rank() + (-ecc).rank()
    return [int(i) for i in score.sort_values().index[:k]]


def max_drawdown(x):
    """Maximum drawdown (negative number) of a return array or matrix (per column)."""
    w = np.cumprod(1.0 + np.asarray(x), axis=0)
    peak = np.maximum.accumulate(w, axis=0)
    return (w / peak - 1.0).min(axis=0)


def pca_kmeans_select(R_df, k, threshold=0.5, per_cluster=2, seed=0):
    """Re-implementation of the PCA-KMeans stratified selection (Vo, Wang & Nguyen, 2026).

    Features: 3-month return, Sharpe, 60-day volatility (stability), max drawdown,
    all computed over the last year of the look-back. Min-max scaled, risk
    features flipped, PC1 loadings (absolute, normalized) used as weights;
    stocks with score > threshold are clustered with K-Means and the top
    `per_cluster` by score are taken from each cluster.
    """
    X = R_df.values[-TRADING_DAYS:]
    ret3m = np.prod(1.0 + X[-63:], axis=0) - 1.0
    sd = X.std(axis=0, ddof=1) + 1e-12
    sharpe = X.mean(axis=0) / sd * np.sqrt(TRADING_DAYS)
    vol60 = X[-60:].std(axis=0, ddof=1)
    mdd = -max_drawdown(X)  # positive size of drawdown
    F = np.column_stack([ret3m, sharpe, vol60, mdd])
    mn, mx = F.min(axis=0), F.max(axis=0)
    F = (F - mn) / np.where(mx - mn > 1e-12, mx - mn, 1.0)
    F[:, 2] = 1.0 - F[:, 2]
    F[:, 3] = 1.0 - F[:, 3]
    w = np.abs(PCA(n_components=1).fit(F).components_[0])
    w = w / w.sum()
    score = F @ w
    good = np.where(score > threshold)[0]
    n_clusters = int(math.ceil(k / per_cluster))
    sel = []
    if len(good) > n_clusters:
        lab = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(F[good])
        for c in range(n_clusters):
            idx = good[lab == c]
            sel += [int(i) for i in idx[np.argsort(-score[idx])][:per_cluster]]
    else:
        sel = [int(i) for i in good]
    if len(sel) < k:
        rest = [int(i) for i in np.argsort(-score) if i not in sel]
        sel += rest[: k - len(sel)]
    return sel[:k]


In [ ]:
def hrp_weights(cov):
    """Hierarchical Risk Parity (Lopez de Prado, 2016) on a covariance matrix."""
    cov = np.asarray(cov)
    sd = np.sqrt(np.diag(cov))
    C = cov / np.outer(sd, sd)
    D = corr_distance(np.clip(C, -1, 1))
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="single")
    # quasi-diagonal order
    N = cov.shape[0]
    order = [int(Z[-1, 0]), int(Z[-1, 1])]
    while max(order) >= N:
        new = []
        for i in order:
            if i >= N:
                r = int(i - N)
                new += [int(Z[r, 0]), int(Z[r, 1])]
            else:
                new.append(i)
        order = new
    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        nxt = []
        for c in clusters:
            if len(c) <= 1:
                continue
            h = len(c) // 2
            c1, c2 = c[:h], c[h:]

            def cvar(ix):
                sub = cov[np.ix_(ix, ix)]
                iv = 1.0 / np.diag(sub)
                iv /= iv.sum()
                return float(iv @ sub @ iv)

            v1, v2 = cvar(c1), cvar(c2)
            alpha = 1.0 - v1 / (v1 + v2)
            w[c1] *= alpha
            w[c2] *= 1.0 - alpha
            nxt += [c1, c2]
        clusters = nxt
    w = w.sort_index().values
    return w / w.sum()


def minvar_weights(cov, max_w=1.0):
    """Long-only minimum-variance weights (fully invested), solved with SLSQP."""
    n = cov.shape[0]
    x0 = np.ones(n) / n
    res = minimize(lambda w: w @ cov @ w, x0, jac=lambda w: 2 * cov @ w,
                   bounds=[(0.0, max_w)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
                   method="SLSQP", options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x if res.success else x0, 0, None)
    return w / w.sum()


def compute_weights(scheme, R_sel):
    """Portfolio weights for the selected assets, estimated on the look-back only."""
    n = R_sel.shape[1]
    if scheme == "ew" or n == 1:
        return np.ones(n) / n
    X = R_sel[-TRADING_DAYS:]
    if scheme == "invvol":
        iv = 1.0 / (X.std(axis=0, ddof=1) + 1e-12)
        return iv / iv.sum()
    cov = LedoitWolf().fit(X).covariance_
    if scheme == "minvar":
        return minvar_weights(cov)
    if scheme == "hrp":
        return hrp_weights(cov)
    raise ValueError(scheme)


In [ ]:
@dataclass
class Config:
    k: int = 10                  # number of assets to select
    lookback: int = 1260         # look-back in trading days (5 years)
    rebalance_every: int = 63    # trading days between rebalances (quarterly)
    stress_q: float = 0.10       # stress days = market in bottom 10% of the sub-window
    sub_len: int = 504           # sub-window length for robust aggregation (2 years)
    sub_step: int = 126          # step between sub-windows (6 months)
    agg_q: float = 0.75          # pessimistic quantile across sub-windows
    theta: float = 0.5           # quality-diversity trade-off
    kernel_a: float = 0.5        # affine kernel parameter
    tail_u: float = 0.10         # tail level for empirical tail dependence
    min_coverage: float = 0.95   # min share of non-missing days in the look-back
    cost_bps: float = 10.0       # one-way transaction cost (bps of traded value)
    test_start: str = "1990-01-01"
    test_end: str = None
    n_random: int = 20           # random-k replications


class Context:
    """Everything a selection method may use at one rebalance date (look-back data only).
    Expensive estimates are computed once and shared across methods."""

    def __init__(self, R_df, mkt, cfg, date=None):
        self.date = date
        self.R_df = R_df
        self.R = R_df.values
        self.mkt = np.asarray(mkt)
        self.cfg = cfg
        self.tickers = list(R_df.columns)

    @cached_property
    def z(self):
        return quality_scores(self.R_df).values

    @cached_property
    def C_plain(self):
        return shrunk_corr(self.R)

    @cached_property
    def C_robust(self):
        c = self.cfg
        return robust_stress_corr(self.R, self.mkt, c.stress_q, c.sub_len, c.sub_step, c.agg_q)[0]

    @cached_property
    def C_stress_single(self):
        c = self.cfg
        n = len(self.mkt)
        return robust_stress_corr(self.R, self.mkt, c.stress_q, n, n, 0.5)[0]

    @cached_property
    def C_plain_robust(self):
        """Robust aggregation across sub-windows but on ALL days (no stress conditioning)."""
        c = self.cfg
        return robust_stress_corr(self.R, self.mkt, 1.0, c.sub_len, c.sub_step, c.agg_q)[0]

    @cached_property
    def taildep(self):
        return lower_tail_dependence(self.R, self.cfg.tail_u)


# Each selection method: f(ctx) -> list of column indices.
def m_srdas(ctx, theta=None):
    th = ctx.cfg.theta if theta is None else theta
    return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, th, ctx.cfg.kernel_a)


def m_srdas_noq(ctx):
    return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, 0.0, ctx.cfg.kernel_a)


def m_das_plain(ctx):
    return logdet_select(ctx.C_plain, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_das_plain_robust(ctx):
    return logdet_select(ctx.C_plain_robust, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_das_stress_single(ctx):
    return logdet_select(ctx.C_stress_single, ctx.z, ctx.cfg.k, ctx.cfg.theta, ctx.cfg.kernel_a)


def m_stress_cluster(ctx):
    return cluster_pick(corr_distance(ctx.C_robust), ctx.z, ctx.cfg.k)


def m_corr_cluster(ctx):
    return cluster_pick(corr_distance(ctx.C_plain), ctx.z, ctx.cfg.k)


def m_taildep_cluster(ctx):
    return cluster_pick(1.0 - ctx.taildep, ctx.z, ctx.cfg.k)


def m_mst_periphery(ctx):
    return mst_periphery_select(ctx.C_plain, ctx.cfg.k)


def m_pca_kmeans(ctx):
    return pca_kmeans_select(ctx.R_df, ctx.cfg.k)


def m_topk_quality(ctx):
    return [int(i) for i in np.argsort(-ctx.z)[: ctx.cfg.k]]


def make_random(seed):
    def m_random(ctx):
        day = ctx.date.toordinal() if ctx.date is not None else 0
        rng = np.random.default_rng([seed, day])
        return [int(i) for i in rng.choice(len(ctx.tickers), size=min(ctx.cfg.k, len(ctx.tickers)), replace=False)]
    return m_random


def m_all(ctx):
    return list(range(len(ctx.tickers)))


@dataclass
class Strategy:
    name: str
    select: callable
    weights: str = "ew"
    group: str = ""


def default_strategies(cfg, weights="ew", include_random=True):
    """The full comparison set: proposed method, ablations, baselines, benchmarks."""
    S = [
        Strategy("SR-DAS", m_srdas, weights, "Proposed"),
        # ablations (each removes one component of SR-DAS)
        Strategy("Abl: Stress-Cluster (rule)", m_stress_cluster, weights, "Ablation"),
        Strategy("Abl: DAS all-days (stress)", m_das_plain_robust, weights, "Ablation"),
        Strategy("Abl: DAS stress 1-window (robust)", m_das_stress_single, weights, "Ablation"),
        Strategy("Abl: SR-DAS theta=0 (quality)", m_srdas_noq, weights, "Ablation"),
        Strategy("Abl: DAS plain corr (all)", m_das_plain, weights, "Ablation"),
        # literature baselines
        Strategy("TailDep-Cluster", m_taildep_cluster, weights, "Baseline"),
        Strategy("Corr-Cluster", m_corr_cluster, weights, "Baseline"),
        Strategy("MST-Periphery", m_mst_periphery, weights, "Baseline"),
        Strategy("PCA-KMeans (Vo et al.)", m_pca_kmeans, weights, "Baseline"),
        Strategy("Top-k Quality", m_topk_quality, weights, "Baseline"),
        # full-universe benchmarks (not selection methods)
        Strategy("EW-All (1/N)", m_all, "ew", "Benchmark"),
        Strategy("HRP-All", m_all, "hrp", "Benchmark"),
    ]
    if include_random:
        S += [Strategy(f"Random-k #{s}", make_random(s), weights, "Random") for s in range(cfg.n_random)]
    return S


@dataclass
class Result:
    name: str
    group: str
    returns: pd.Series                      # daily net portfolio returns
    gross_returns: pd.Series
    turnover: pd.Series                     # one-way turnover at each rebalance
    holdings: dict = field(default_factory=dict)   # rebalance date -> {ticker: weight}
    diag: pd.DataFrame = None               # per-period diagnostics


def drifted_weights(w, R_hold):
    """Buy-and-hold weights at the end of a holding period."""
    v = w * np.prod(1.0 + R_hold, axis=0)
    s = v.sum()
    return v / s if s > 0 else w


def avg_pairwise_corr(X):
    sd = X.std(axis=0, ddof=1)
    ok = sd > 1e-12
    if ok.sum() < 2 or X.shape[0] < 5:
        return np.nan
    C = np.corrcoef(X[:, ok], rowvar=False)
    return (C.sum() - np.trace(C)) / (C.shape[0] * (C.shape[0] - 1))


def period_diagnostics(R_hold, w):
    """Out-of-sample diversification of the held basket over one holding period."""
    n = R_hold.shape[1]
    if n < 2 or R_hold.shape[0] < 5:
        return np.nan, np.nan
    sd = R_hold.std(axis=0, ddof=1)
    ok = sd > 1e-12
    if ok.sum() < 2:
        return np.nan, np.nan
    C = np.corrcoef(R_hold[:, ok], rowvar=False)
    avg_corr = (C.sum() - np.trace(C)) / (C.shape[0] * (C.shape[0] - 1))
    port_sd = (R_hold @ w).std(ddof=1)
    dr = float(w @ sd / port_sd) if port_sd > 1e-12 else np.nan
    return avg_corr, dr


def run_backtest(returns, market, strategies, cfg, universe_fn=None, verbose=True, eval_stress_q=0.10):
    """Walk-forward backtest.

    At each rebalance date t (every cfg.rebalance_every days):
      * eligible assets = universe_fn(t) with >= min_coverage data in the look-back
        and a valid return on day t;
      * each strategy selects/weights using ONLY data in (t-lookback, t];
      * portfolios are held buy-and-hold from t+1 to the next rebalance.
    Missing returns while held (e.g. delisting) are treated as 0 (cash).
    Transaction cost = cost_bps * one-way turnover, charged on the first day.
    """
    returns = returns.sort_index()
    market = market.reindex(returns.index)
    dates = returns.index
    t0 = dates.searchsorted(pd.Timestamp(cfg.test_start))
    t0 = max(t0, cfg.lookback)
    t_end = len(dates) if cfg.test_end is None else dates.searchsorted(pd.Timestamp(cfg.test_end), side="right")
    rebal_idx = list(range(t0 - 1, t_end - 1, cfg.rebalance_every))
    # Evaluation-only threshold for "stress days" in the test period (never used for selection)
    eval_thr = market.iloc[t0:t_end].quantile(eval_stress_q)

    out = {s.name: dict(net=[], gross=[], turn={}, hold={}, diag=[], prev=None) for s in strategies}
    tic = time.time()
    for n_r, t in enumerate(rebal_idx):
        t_next = min(t + cfg.rebalance_every, t_end - 1)
        if t_next <= t:
            break
        win = returns.iloc[t - cfg.lookback + 1: t + 1]
        mkt_win = market.iloc[t - cfg.lookback + 1: t + 1]
        cols = list(win.columns) if universe_fn is None else [c for c in universe_fn(dates[t]) if c in win.columns]
        w_df = win[cols]
        cov_ok = w_df.notna().mean() >= cfg.min_coverage
        last_ok = w_df.iloc[-1].notna()
        elig = [c for c in cols if cov_ok[c] and last_ok[c]]
        mk_ok = mkt_win.notna().values
        R_df = w_df.loc[mk_ok, elig].fillna(0.0)
        ctx = Context(R_df, mkt_win.values[mk_ok], cfg, dates[t])
        hold = returns.iloc[t + 1: t_next + 1]
        hold_stress = (market.iloc[t + 1: t_next + 1] <= eval_thr).values
        for s in strategies:
            o = out[s.name]
            idx = s.select(ctx)
            tick = [elig[i] for i in idx]
            w = compute_weights(s.weights, R_df.values[:, idx])
            R_hold = hold[tick].fillna(0.0).values
            # buy-and-hold path
            growth = np.cumprod(1.0 + R_hold, axis=0) * w
            val = np.concatenate([[1.0], growth.sum(axis=1)])
            r_gross = val[1:] / val[:-1] - 1.0
            # turnover vs drifted previous portfolio
            new = pd.Series(w, index=tick)
            prev = o["prev"]
            if prev is None:
                turn = 1.0
            else:
                u = new.index.union(prev.index)
                turn = 0.5 * float(np.abs(new.reindex(u, fill_value=0) - prev.reindex(u, fill_value=0)).sum())
            r_net = r_gross.copy()
            r_net[0] -= turn * cfg.cost_bps / 1e4
            o["gross"].append(pd.Series(r_gross, index=hold.index))
            o["net"].append(pd.Series(r_net, index=hold.index))
            o["turn"][dates[t]] = turn
            o["hold"][dates[t]] = dict(zip(tick, np.round(w, 6)))
            o["prev"] = pd.Series(drifted_weights(w, R_hold), index=tick)
            ac, dr = period_diagnostics(R_hold, w)
            n_st = int(hold_stress.sum())
            ac_st = avg_pairwise_corr(R_hold[hold_stress]) if n_st >= 5 else np.nan
            o["diag"].append(dict(date=dates[t], n_eligible=len(elig), n_held=len(tick),
                                  avg_corr_oos=ac, div_ratio_oos=dr,
                                  stress_corr_oos=ac_st, n_stress_days=n_st))
        if verbose and (n_r % 10 == 0 or n_r == len(rebal_idx) - 1):
            print(f"  rebalance {n_r + 1}/{len(rebal_idx)}  {dates[t].date()}  eligible={len(elig)}  "
                  f"elapsed={time.time() - tic:.0f}s")
    results = {}
    for s in strategies:
        o = out[s.name]
        results[s.name] = Result(s.name, s.group, pd.concat(o["net"]), pd.concat(o["gross"]),
                                 pd.Series(o["turn"]), o["hold"], pd.DataFrame(o["diag"]).set_index("date"))
    return results


In [ ]:
def perf_metrics(r, rf=None):
    """Standard performance metrics of a daily return series."""
    r = r.dropna()
    rf_ = pd.Series(0.0, index=r.index) if rf is None else rf.reindex(r.index).fillna(0.0)
    ex = r - rf_
    yrs = len(r) / TRADING_DAYS
    wealth = (1 + r).cumprod()
    cagr = wealth.iloc[-1] ** (1 / yrs) - 1 if yrs > 0 else np.nan
    vol = r.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = ex.mean() / ex.std(ddof=1) * np.sqrt(TRADING_DAYS)
    dn = ex[ex < 0]
    sortino = ex.mean() * TRADING_DAYS / (np.sqrt((dn ** 2).sum() / len(ex)) * np.sqrt(TRADING_DAYS))
    mdd = (wealth / wealth.cummax() - 1).min()
    cvar = r[r <= r.quantile(0.05)].mean()
    return dict(CAGR=cagr, Vol=vol, Sharpe=sharpe, Sortino=sortino, MaxDD=mdd,
                Calmar=cagr / abs(mdd) if mdd < 0 else np.nan, CVaR5_daily=cvar)


def stress_stats(r, market, q=0.10):
    """Out-of-sample behaviour on market stress days (bottom q of the market in the window)."""
    df = pd.concat([r, market], axis=1, keys=["p", "m"], sort=True).dropna()
    st = df[df["m"] <= df["m"].quantile(q)]
    beta = np.cov(st["p"], st["m"])[0, 1] / st["m"].var() if len(st) > 5 else np.nan
    return dict(StressBeta=beta, StressRet_daily=st["p"].mean())


def summary_table(results, rf=None, start=None, end=None, collapse_random=True, market=None):
    """One row per strategy; Random-k replications are averaged into one row.

    If `market` is given, adds out-of-sample stress diagnostics: beta and mean return on
    market stress days, and the average pairwise correlation of the held basket on
    stress days (weighted by number of stress days per holding period)."""
    rows = {}
    for name, res in results.items():
        r = res.returns.loc[start:end]
        if len(r) < 20:
            continue
        m = perf_metrics(r, rf)
        yrs = len(r) / TRADING_DAYS
        tv = res.turnover.loc[start:end] if start or end else res.turnover
        m["Turnover/yr"] = tv.sum() / yrs if yrs > 0 else np.nan
        d = res.diag.loc[start:end] if res.diag is not None else None
        m["AvgCorr_OOS"] = d["avg_corr_oos"].mean() if d is not None else np.nan
        m["DivRatio_OOS"] = d["div_ratio_oos"].mean() if d is not None else np.nan
        if d is not None and "stress_corr_oos" in d:
            ok = d["stress_corr_oos"].notna()
            m["StressCorr_OOS"] = np.average(d.loc[ok, "stress_corr_oos"], weights=d.loc[ok, "n_stress_days"]) if ok.any() else np.nan
        if market is not None:
            m.update(stress_stats(r, market.loc[start:end]))
        m["Group"] = res.group
        rows[name] = m
    df = pd.DataFrame(rows).T
    if collapse_random:
        rnd = [n for n in df.index if n.startswith("Random-k #")]
        if rnd:
            num = df.loc[rnd].drop(columns="Group").astype(float).mean()
            num["Group"] = "Random"
            df = df.drop(index=rnd)
            df.loc[f"Random-k (mean of {len(rnd)})"] = num
    order = {"Proposed": 0, "Proposed-2": 0.5, "Ablation": 1, "Original": 1.5, "Baseline": 2, "Random": 3, "Benchmark": 4}
    df["_o"] = df["Group"].map(order)
    df = df.sort_values(["_o"], kind="stable").drop(columns="_o")
    cols = ["Group", "CAGR", "Vol", "Sharpe", "Sortino", "MaxDD", "Calmar", "CVaR5_daily",
            "Turnover/yr", "AvgCorr_OOS", "StressCorr_OOS", "DivRatio_OOS", "StressBeta", "StressRet_daily"]
    return df[[c for c in cols if c in df.columns]]


def format_table(df):
    fmt = {"CAGR": "{:.2%}", "Vol": "{:.2%}", "Sharpe": "{:.3f}", "Sortino": "{:.3f}",
           "MaxDD": "{:.2%}", "Calmar": "{:.3f}", "CVaR5_daily": "{:.2%}", "Turnover/yr": "{:.2f}",
           "AvgCorr_OOS": "{:.3f}", "DivRatio_OOS": "{:.3f}", "StressCorr_OOS": "{:.3f}",
           "StressBeta": "{:.3f}", "StressRet_daily": "{:.2%}"}
    out = df.copy()
    for c, f in fmt.items():
        if c in out:
            out[c] = out[c].map(lambda v: f.format(v) if pd.notna(v) else "")
    return out


def crisis_table(results, windows, market=None):
    """Cumulative return and max drawdown of each strategy inside crisis windows."""
    rows = []
    series = {n: r.returns for n, r in results.items() if not n.startswith("Random-k #")}
    rnd = [r.returns for n, r in results.items() if n.startswith("Random-k #")]
    if rnd:
        series[f"Random-k (mean of {len(rnd)})"] = pd.concat(rnd, axis=1, sort=True).mean(axis=1)
    if market is not None:
        series["Market"] = market
    for label, (a, b) in windows.items():
        for n, r in series.items():
            x = r.loc[a:b].dropna()
            if len(x) < 5:
                continue
            w = (1 + x).cumprod()
            rows.append(dict(Crisis=label, Strategy=n, Return=w.iloc[-1] - 1,
                             MaxDD=(w / w.cummax() - 1).min()))
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    return df.pivot(index="Strategy", columns="Crisis", values="MaxDD"), \
        df.pivot(index="Strategy", columns="Crisis", values="Return")


def _sharpe(x):
    s = x.std(axis=0, ddof=1)
    return x.mean(axis=0) / np.where(s > 0, s, np.nan)


def sharpe_test(a, b, rf=None, block=21, B=2000, seed=0):
    """Test H0: Sharpe(a) = Sharpe(b) for two daily return series.

    * Jobson-Korkie test with Memmel (2003) correction (assumes iid normal).
    * Circular block bootstrap (block length `block`) of the Sharpe difference,
      centred under H0 — robust to autocorrelation and fat tails, in the
      spirit of Ledoit & Wolf (2008).
    Returns annualized difference and both p-values.
    """
    df = pd.concat([a, b], axis=1, sort=True).dropna()
    if rf is not None:
        r = rf.reindex(df.index).fillna(0.0).values[:, None]
        X = df.values - r
    else:
        X = df.values
    T = len(X)
    sr = _sharpe(X)
    diff = (sr[0] - sr[1]) * np.sqrt(TRADING_DAYS)
    rho = np.corrcoef(X.T)[0, 1]
    var_jk = (2 * (1 - rho) + 0.5 * (sr[0] ** 2 + sr[1] ** 2 - 2 * sr[0] * sr[1] * rho ** 2)) / T
    p_jk = 2 * (1 - norm.cdf(abs(sr[0] - sr[1]) / np.sqrt(var_jk)))
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    d_obs = sr[0] - sr[1]
    boots = np.empty(B)
    for i in range(B):
        st = rng.integers(0, T, nb)
        idx = ((st[:, None] + np.arange(block)[None, :]) % T).ravel()[:T]
        s = _sharpe(X[idx])
        boots[i] = s[0] - s[1]
    p_bs = float(np.mean(np.abs(boots - d_obs) >= abs(d_obs)))
    return dict(SharpeDiff_ann=diff, p_JK_Memmel=p_jk, p_block_bootstrap=p_bs)


def sharpe_tests_vs(results, ref, rf=None, B=2000):
    rows = {}
    for n, r in results.items():
        if n == ref or n.startswith("Random-k #"):
            continue
        rows[n] = sharpe_test(results[ref].returns, r.returns, rf, B=B)
    return pd.DataFrame(rows).T


def selection_frequency(result):
    """How often each asset was held across rebalances."""
    cnt = {}
    for h in result.holdings.values():
        for t in h:
            cnt[t] = cnt.get(t, 0) + 1
    return pd.Series(cnt).sort_values(ascending=False) / max(1, len(result.holdings))


In [ ]:
import matplotlib.pyplot as plt

PALETTE = {"Proposed": "#1f4e9c", "Proposed-2": "#2a9d8f", "Original": "#e9a23b", "Ablation": "#8fb0de", "Baseline": "#d1495b",
           "Random": "#9a9a9a", "Benchmark": "#2b2b2b"}


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


def plot_wealth(results, names, title="Cumulative wealth (log scale, net of costs)"):
    fig, ax = plt.subplots(figsize=(11, 5))
    for n in names:
        r = results[n]
        w = (1 + r.returns).cumprod()
        lw = 2.4 if r.group == "Proposed" else 1.2
        ax.plot(w.index, w.values, lw=lw, label=n,
                color=PALETTE.get(r.group) if r.group in ("Proposed", "Benchmark") else None)
    ax.set_yscale("log"); ax.set_title(title, loc="left"); ax.legend(fontsize=8, ncol=2, frameon=False)
    _style(ax); plt.tight_layout(); plt.show()


def plot_drawdowns(results, names, title="Drawdowns"):
    fig, ax = plt.subplots(figsize=(11, 4))
    for n in names:
        r = results[n]
        w = (1 + r.returns).cumprod()
        ax.plot(w.index, (w / w.cummax() - 1).values, lw=2 if r.group == "Proposed" else 1, label=n)
    ax.set_title(title, loc="left"); ax.legend(fontsize=8, ncol=2, frameon=False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    _style(ax); plt.tight_layout(); plt.show()


def plot_metric_bars(table, metric, title=None, higher_is_better=True):
    t = table[metric].astype(float).sort_values(ascending=not higher_is_better)
    colors = [PALETTE.get(g, "#888") for g in table.loc[t.index, "Group"]]
    fig, ax = plt.subplots(figsize=(9, 0.35 * len(t) + 1))
    ax.barh(t.index[::-1], t.values[::-1], color=colors[::-1])
    ax.set_title(title or metric, loc="left"); _style(ax); ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_sensitivity(sens, metrics=("Sharpe", "MaxDD", "StressCorr_OOS")):
    params = sens["param"].unique()
    fig, axes = plt.subplots(len(metrics), len(params), figsize=(3.2 * len(params), 2.6 * len(metrics)), squeeze=False)
    for j, p in enumerate(params):
        d = sens[sens["param"] == p]
        for i, m in enumerate(metrics):
            ax = axes[i, j]
            ax.plot(d["value"].astype(str), d[m].astype(float), marker="o", color=PALETTE["Proposed"])
            if i == 0: ax.set_title(p)
            if j == 0: ax.set_ylabel(m)
            _style(ax)
    plt.tight_layout(); plt.show()


In [ ]:
from scipy.linalg import pinvh
from scipy.stats import wilcoxon

THETA_LOG = {}   # rebalance date -> balanced theta chosen by the primary method (for reporting)


def uniqueness_scores(K):
    """Diversity contribution of each asset: log of its conditional variance given all
    other assets under kernel K, i.e. log det(K) - log det(K without i) = -log (K^-1)_ii.
    These are the canonical per-asset diversity scores of the log-det objective."""
    Kinv = pinvh(K)
    return -np.log(np.clip(np.diag(Kinv), 1e-12, None))


def balanced_theta(K, z, beta=1.0):
    """Scale rule that fixes the quality/diversity imbalance of the original SR-DAS.

    The objective is  2*theta*sum(z_S) + log det K_S. theta is set at every rebalance so
    that the cross-sectional spread of the quality term (2*theta*z_i) equals the spread
    of the diversity scores (uniqueness_scores). beta = 1 gives both terms equal weight.
    This is a fixed rule — nothing is tuned on returns."""
    g = uniqueness_scores(K)
    return beta * np.std(g) / (2.0 * np.std(z) + 1e-12)


def make_balanced(corr_attr, log=False):
    """Balanced SR-DAS on a given dependence estimate (attribute name of Context)."""
    def m(ctx):
        C = getattr(ctx, corr_attr)
        K = affine_corr_kernel(C, ctx.cfg.kernel_a)
        th = balanced_theta(K, ctx.z)
        if log and ctx.date is not None:
            THETA_LOG[ctx.date] = th
        return logdet_select(C, ctx.z, ctx.cfg.k, th, ctx.cfg.kernel_a)
    return m


def make_screen(corr_attr, frac=0.5):
    """Screen-then-diversify: keep the top `frac` of assets by quality, then pure
    log-det diversity (theta = 0) among them on the given dependence estimate."""
    def m(ctx):
        C = getattr(ctx, corr_attr)
        n = len(ctx.z)
        keep = np.argsort(-ctx.z)[: max(ctx.cfg.k, int(np.ceil(frac * n)))]
        keep = np.sort(keep)
        sub = logdet_select(C[np.ix_(keep, keep)], ctx.z[keep], ctx.cfg.k, 0.0, ctx.cfg.kernel_a)
        return [int(keep[i]) for i in sub]
    return m


def make_theta(theta):
    """SR-DAS with a fixed theta (used for the trade-off frontier)."""
    def m(ctx):
        return logdet_select(ctx.C_robust, ctx.z, ctx.cfg.k, theta, ctx.cfg.kernel_a)
    return m


def v3_strategies(cfg, include_random=True, weights="ew"):
    """Pre-specified comparison set for the revised study."""
    S = [
        Strategy("B-SR-DAS (balanced)", make_balanced("C_robust", log=True), weights, "Proposed"),
        Strategy("S-SR-DAS (screen+diversify)", make_screen("C_robust"), weights, "Proposed-2"),
        # ablations of the PRIMARY method: one ingredient changed each
        Strategy("Abl: B-DAS all-days (stress)", make_balanced("C_plain_robust"), weights, "Ablation"),
        Strategy("Abl: B-DAS stress 1-window (robust)", make_balanced("C_stress_single"), weights, "Ablation"),
        Strategy("Abl: Stress-Cluster (rule)", m_stress_cluster, weights, "Ablation"),
        # the original specification, for reference
        Strategy("Original SR-DAS (theta=0.5)", m_srdas, weights, "Original"),
        Strategy("Original SR-DAS (theta=0)", m_srdas_noq, weights, "Original"),
        # literature baselines and benchmarks
        Strategy("TailDep-Cluster", m_taildep_cluster, weights, "Baseline"),
        Strategy("Corr-Cluster", m_corr_cluster, weights, "Baseline"),
        Strategy("MST-Periphery", m_mst_periphery, weights, "Baseline"),
        Strategy("PCA-KMeans (Vo et al.)", m_pca_kmeans, weights, "Baseline"),
        Strategy("Top-k Quality", m_topk_quality, weights, "Baseline"),
        Strategy("EW-All (1/N)", m_all, "ew", "Benchmark"),
        Strategy("HRP-All", m_all, "hrp", "Benchmark"),
    ]
    if include_random:
        S += [Strategy(f"Random-k #{s}", make_random(s), weights, "Random") for s in range(cfg.n_random)]
    return S


# ---------- statistical helpers for the pre-specified hypotheses ----------
def paired_period_test(res_a, res_b, col="stress_corr_oos", alternative="less"):
    """Paired test across holding periods of a per-period diagnostic (e.g. the held
    basket's stress-day correlation). Wilcoxon signed-rank on period-by-period
    differences a - b. alternative='less' tests whether a is systematically lower."""
    d = (res_a.diag[col] - res_b.diag[col]).dropna()
    if len(d) < 6 or np.allclose(d, 0):
        return dict(n_periods=len(d), mean_diff=float(d.mean()) if len(d) else np.nan,
                    share_a_lower=float((d < 0).mean()) if len(d) else np.nan, p_value=np.nan)
    p = wilcoxon(d, alternative=alternative).pvalue
    return dict(n_periods=len(d), mean_diff=float(d.mean()), share_a_lower=float((d < 0).mean()), p_value=float(p))


def ratio_ci(a, b, stat="vol", block=21, B=2000, seed=0, q=0.10, market=None):
    """Block-bootstrap 95% CI for a risk ratio of two daily return series:
    stat='vol' -> vol(a)/vol(b);  stat='cvar' -> CVaR5(a)/CVaR5(b)."""
    X = pd.concat([a, b], axis=1, sort=True).dropna().values
    T = len(X)

    def f(Y):
        if stat == "vol":
            s = Y.std(axis=0, ddof=1)
            return s[0] / s[1]
        lo = [Y[Y[:, j] <= np.quantile(Y[:, j], 0.05), j].mean() for j in range(2)]
        return lo[0] / lo[1]

    obs = f(X)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    bs = np.empty(B)
    for i in range(B):
        idx = ((rng.integers(0, T, nb)[:, None] + np.arange(block)[None, :]) % T).ravel()[:T]
        bs[i] = f(X[idx])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(ratio=float(obs), ci_low=float(lo), ci_high=float(hi))


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (controls family-wise error)."""
    p = pd.Series(pvals, dtype=float)
    ok = p.dropna().sort_values()
    m = len(ok)
    adj = pd.Series(np.nan, index=p.index)
    running = 0.0
    for i, (k, v) in enumerate(ok.items()):
        running = max(running, min(1.0, (m - i) * v))
        adj[k] = running
    return adj


def sharpe_tests_holm(results, ref, rf=None, B=2000):
    t = sharpe_tests_vs(results, ref, rf, B=B)
    t["p_holm (bootstrap)"] = holm(t["p_block_bootstrap"])
    return t


def evaluate_hypotheses(res, primary, rf, B=2000, core_only=False):
    """Evaluate the pre-registered hypotheses for one method on one sample.
    H1-H4 apply to any method; H5-H7 (ablations) apply to the primary method only
    (core_only=True skips them). Returns: hypothesis, statistic, p-value / CI, verdict."""
    rows = []

    def add(h, desc, stat, detail, ok):
        rows.append(dict(Hypothesis=h, Description=desc, Statistic=stat, Detail=detail,
                         Verdict="SUPPORTED" if ok else "NOT supported"))

    P = res[primary]
    # H1: lower stress-day correlation of the held basket than Top-k Quality
    t = paired_period_test(P, res["Top-k Quality"])
    add("H1", "Stress-day basket correlation lower than Top-k Quality (paired, one-sided)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H2: lower volatility than 1/N
    c = ratio_ci(P.returns, res["EW-All (1/N)"].returns, "vol", B=B)
    add("H2", "Volatility lower than EW-All (1/N)", f"ratio {c['ratio']:.3f}",
        f"95% CI [{c['ci_low']:.3f}, {c['ci_high']:.3f}]", c["ci_high"] < 1)
    # H3: lower volatility than pure diversity (theta = 0) -> the balance works
    c = ratio_ci(P.returns, res["Original SR-DAS (theta=0)"].returns, "vol", B=B)
    add("H3", "Volatility lower than pure-diversity SR-DAS (theta=0)", f"ratio {c['ratio']:.3f}",
        f"95% CI [{c['ci_low']:.3f}, {c['ci_high']:.3f}]", c["ci_high"] < 1)
    # H4: no significant Sharpe under-performance vs 1/N
    s = sharpe_test(P.returns, res["EW-All (1/N)"].returns, rf, B=B)
    add("H4", "Sharpe not significantly below EW-All (1/N) (two-sided, 5%)",
        f"diff {s['SharpeDiff_ann']:+.3f}", f"p = {s['p_block_bootstrap']:.4f}",
        not (s["SharpeDiff_ann"] < 0 and s["p_block_bootstrap"] < 0.05))
    if core_only:
        return pd.DataFrame(rows).set_index("Hypothesis")
    # H5: lower stress-day correlation than cluster-then-pick on the same matrix (RQ1)
    t = paired_period_test(P, res["Abl: Stress-Cluster (rule)"])
    add("H5", "Stress-day basket correlation lower than Stress-Cluster rule (RQ1, one-sided)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H6: stress conditioning lowers realized stress correlation (RQ2)
    t = paired_period_test(P, res["Abl: B-DAS all-days (stress)"])
    add("H6", "Stress conditioning lowers stress-day basket correlation vs all-day correlation (RQ2)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    # H7: robust aggregation lowers realized stress correlation (RQ3)
    t = paired_period_test(P, res["Abl: B-DAS stress 1-window (robust)"])
    add("H7", "Robust aggregation lowers stress-day basket correlation vs single window (RQ3)",
        f"mean diff {t['mean_diff']:+.3f}", f"p = {t['p_value']:.4f}, lower in {t['share_a_lower']:.0%} of periods",
        t["p_value"] < 0.05)
    return pd.DataFrame(rows).set_index("Hypothesis")


def run_frontier(returns, market, cfg, thetas, rf=None):
    """SR-DAS for a grid of fixed theta values: maps the quality-diversity trade-off."""
    S = [Strategy(f"theta={t:g}", make_theta(t), "ew", "Proposed") for t in thetas]
    r = run_backtest(returns, market, S, cfg, verbose=False)
    tab = summary_table(r, rf, market=market)
    tab["theta"] = thetas
    return tab


def plot_frontier(frontier, points, x="StressCorr_OOS", y="Vol", title=None):
    """Frontier of fixed-theta SR-DAS (line) with other strategies as labelled points."""
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    f = frontier.sort_values("theta")
    ax.plot(f[x].astype(float), f[y].astype(float), "-o", color=PALETTE["Proposed"], lw=1.5, ms=4,
            label="SR-DAS, fixed theta")
    for _, r in f.iterrows():
        ax.annotate(f"θ={r['theta']:g}", (float(r[x]), float(r[y])), fontsize=7, xytext=(4, 2),
                    textcoords="offset points", color=PALETTE["Proposed"])
    for name, r in points.iterrows():
        g = r["Group"]
        ax.scatter(float(r[x]), float(r[y]), s=60 if g in ("Proposed", "Proposed-2") else 30,
                   color=PALETTE.get(g, "#888"), zorder=3,
                   marker="*" if g in ("Proposed", "Proposed-2") else "o")
        ax.annotate(name, (float(r[x]), float(r[y])), fontsize=7, xytext=(4, -9), textcoords="offset points")
    ax.set_xlabel(f"{x} (lower = more diversified in stress)")
    ax.set_ylabel(f"{y} (lower = better)")
    if y in ("Vol",):
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title or f"Trade-off: {x} vs {y}", loc="left")
    _style(ax); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


### Manuscript helpers
Publication style (validated colour-blind-safe palette, with line styles and direct labels as secondary encoding), figure/table export, cost re-pricing from gross returns, and block-bootstrap Sharpe CIs.

In [ ]:
from dataclasses import replace as dc_replace
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch

# ---------- validated categorical palette (fixed order) + neutral ----------
C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
C_GRAY, C_INK, C_INK2, C_GRID = "#8a8985", "#0b0b0b", "#52514e", "#e4e3df"
# entity -> (color, linestyle); color follows the strategy, never its rank
SERIES_STYLE = {
    "B-SR-DAS (balanced)": (C_BLUE, "-"),
    "Top-k Quality": (C_ORANGE, "--"),
    "TailDep-Cluster": (C_AQUA, "-."),
    "S-SR-DAS (screen+diversify)": (C_VIOLET, ":"),
    "EW-All (1/N)": (C_GRAY, "-"),
}


def pub_style():
    """Journal-style matplotlib defaults: thin marks, recessive grid, readable type."""
    mpl.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
        "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": C_INK2,
        "axes.labelcolor": C_INK, "xtick.color": C_INK2, "ytick.color": C_INK2,
        "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.6, "lines.linewidth": 1.6,
        "legend.frameon": False, "font.family": "DejaVu Sans", "pdf.fonttype": 42,
    })


FIG_DIR = "outputs_manuscript/figures"
TAB_DIR = "outputs_manuscript/tables"


def save_fig(fig, name):
    """Save a figure as 300-dpi PNG and vector PDF."""
    os.makedirs(FIG_DIR, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), bbox_inches="tight")
    plt.show()


def export_table(df, name, caption=None, float_fmt="{:.3f}"):
    """Save a table as CSV and LaTeX (booktabs), and register it for the Excel workbook."""
    os.makedirs(TAB_DIR, exist_ok=True)
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"))
    try:
        tex = df.to_latex(float_format=lambda v: float_fmt.format(v), caption=caption,
                          label=f"tab:{name}", escape=True)
        with open(os.path.join(TAB_DIR, f"{name}.tex"), "w") as f:
            f.write(tex)
    except Exception as e:  # jinja2 missing etc.
        print("  (LaTeX export skipped:", e, ")")
    TABLES[name] = df
    return df


TABLES = {}


def write_workbook(path="outputs_manuscript/tables/all_tables.xlsx"):
    with pd.ExcelWriter(path) as xw:
        for name, df in TABLES.items():
            df.to_excel(xw, sheet_name=name[:31])
    print("Excel workbook:", path)


# ---------- analysis helpers ----------
def apply_costs(res, bps):
    """Re-price a backtest at a different one-way cost (bps) from its gross returns and
    turnover - identical to re-running the backtest with that cost."""
    out = {}
    for n, r in res.items():
        net = r.gross_returns.copy()
        idx = net.index
        for d, turn in r.turnover.items():
            i = idx.searchsorted(d, side="right")
            if i < len(idx):
                net.iloc[i] -= turn * bps / 1e4
        out[n] = dc_replace(r, returns=net)
    return out


def sharpe_ci(r, rf=None, block=21, B=2000, seed=0):
    """Annualized Sharpe ratio with a circular block-bootstrap 95% CI."""
    r = r.dropna()
    x = (r - rf.reindex(r.index).fillna(0.0)).values if rf is not None else r.values
    T = len(x)
    s = x.mean() / x.std(ddof=1) * np.sqrt(TRADING_DAYS)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    bs = np.empty(B)
    for i in range(B):
        idx = ((rng.integers(0, T, nb)[:, None] + np.arange(block)[None, :]) % T).ravel()[:T]
        y = x[idx]
        bs[i] = y.mean() / y.std(ddof=1) * np.sqrt(TRADING_DAYS)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return s, lo, hi


def sharpe_ci_table(res, names, rf, B=2000):
    rows = {}
    for n in names:
        s, lo, hi = sharpe_ci(res[n].returns, rf, B=B)
        rows[n] = dict(Sharpe=s, CI_low=lo, CI_high=hi)
    return pd.DataFrame(rows).T


def avg_corr_matrix(df):
    C = df.corr().values
    n = C.shape[0]
    return (C.sum() - n) / (n * (n - 1))


def descriptive_block(ind, mkt, start, end, label):
    x, m = ind.loc[start:end], mkt.loc[start:end]
    x = x.loc[:, x.notna().mean() > 0.95]
    st = m <= m.quantile(0.10)
    calm = m > m.quantile(0.50)
    ann_ret = (1 + x).prod() ** (TRADING_DAYS / len(x)) - 1
    ann_vol = x.std() * np.sqrt(TRADING_DAYS)
    return {
        "Sample": label, "Start": str(x.index[0].date()), "End": str(x.index[-1].date()),
        "Trading days": len(x), "Assets": x.shape[1],
        "Asset ann. return, median": ann_ret.median(), "Asset ann. vol, median": ann_vol.median(),
        "Asset ann. vol, min": ann_vol.min(), "Asset ann. vol, max": ann_vol.max(),
        "Market ann. return": (1 + m).prod() ** (TRADING_DAYS / len(m)) - 1,
        "Market ann. vol": m.std() * np.sqrt(TRADING_DAYS),
        "Avg pairwise corr, all days": avg_corr_matrix(x),
        "Avg pairwise corr, calm days": avg_corr_matrix(x[calm]),
        "Avg pairwise corr, stress days": avg_corr_matrix(x[st]),
    }


def robustness_rows(res, label, rf, B=1000):
    """Key metrics + H1-H4 verdicts of the primary method in one robustness configuration."""
    tab = summary_table(res, rf, market=MKT)
    hyp = evaluate_hypotheses(res, "B-SR-DAS (balanced)", rf, B=B, core_only=True)
    row = {"Configuration": label}
    for n, short in [("B-SR-DAS (balanced)", "B-SR-DAS"), ("Top-k Quality", "TopK"), ("EW-All (1/N)", "1/N")]:
        row[f"Sharpe {short}"] = tab.loc[n, "Sharpe"]
        row[f"Vol {short}"] = tab.loc[n, "Vol"]
        row[f"StressCorr {short}"] = tab.loc[n, "StressCorr_OOS"]
    row["MaxDD B-SR-DAS"] = tab.loc["B-SR-DAS (balanced)", "MaxDD"]
    for h in ["H1", "H2", "H3", "H4"]:
        row[h] = "yes" if hyp.loc[h, "Verdict"] == "SUPPORTED" else "no"
    return row


# ---------- figures ----------
def _label_end(ax, x, y, text, color):
    ax.annotate(text, (x, y), xytext=(4, 0), textcoords="offset points", fontsize=7.5,
                color=C_INK, va="center")


def fig_framework():
    """Schematic of the B-SR-DAS pipeline."""
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.set_xlim(0, 10); ax.set_ylim(0, 3.6); ax.axis("off")

    def box(x, y, w, h, title, body, color):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                    fc="white", ec=color, lw=1.4))
        ax.text(x + w / 2, y + h - 0.2, title, ha="center", va="top", fontsize=8.5, weight="bold", color=C_INK)
        ax.text(x + w / 2, y + h - 0.52, body, ha="center", va="top", fontsize=7.2, color=C_INK2, linespacing=1.35)

    def arrow(x0, y0, x1, y1):
        ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="-|>", color=C_INK2, lw=1))

    box(0.05, 2.0, 1.9, 1.45, "Look-back data", "5 years of daily\nreturns up to t\n(no look-ahead)", C_GRAY)
    box(2.35, 2.0, 2.3, 1.45, "Stress dependence", "market-stress days only,\noverlapping sub-windows,\npessimistic quantile,\nPSD projection → C", C_AQUA)
    box(2.35, 0.15, 2.3, 1.45, "Quality score z", "low volatility +\n12-1 momentum,\nrank-normalized", C_ORANGE)
    box(5.05, 1.1, 2.25, 1.45, "Balanced objective", "K = ½C + ½·11ᵀ\nmax 2θΣz + log det K_S\nθ: equal spread of quality\nand diversity terms", C_BLUE)
    box(7.7, 1.1, 2.2, 1.45, "Portfolio", "greedy log-det → k assets\nequal weights, held\none quarter, then\nroll forward", C_VIOLET)
    arrow(1.95, 2.72, 2.35, 2.72); arrow(1.95, 2.3, 2.35, 0.95)
    arrow(4.65, 2.6, 5.05, 2.05); arrow(4.65, 0.9, 5.05, 1.5); arrow(7.3, 1.82, 7.7, 1.82)
    ax.set_title("B-SR-DAS: stress-robust, balanced determinantal asset selection (walk-forward)", loc="left")
    return fig


def fig_motivation(ind, mkt, start="1965-01-01"):
    x, m = ind.loc[start:], mkt.loc[start:]
    pts = list(range(126, len(x), 21))
    roll = pd.Series([avg_corr_matrix(x.iloc[i - 126:i].dropna(axis=1, thresh=100)) for i in pts], index=x.index[pts])
    w = (1 + m).cumprod()
    dd = w / w.cummax() - 1
    fig = plt.figure(figsize=(10, 5.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1], height_ratios=[1, 0.7], hspace=0.35, wspace=0.25)
    a1 = fig.add_subplot(gs[0, 0]); a2 = fig.add_subplot(gs[1, 0], sharex=a1); a3 = fig.add_subplot(gs[:, 1])
    a1.plot(roll.index, roll.values, color=C_BLUE, lw=1.2)
    a1.set_title("(a) Rolling 6-month average pairwise correlation of the 49 industries", loc="left")
    a2.fill_between(dd.index, dd.values, 0, color=C_GRAY, alpha=0.6, lw=0)
    a2.set_title("(b) Market drawdown", loc="left")
    a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    rows = []
    for a, b in [("1965", "1979"), ("1980", "1989"), ("1990", "2004"), ("2005", "2026")]:
        xx, mm = x.loc[a:b].dropna(axis=1, thresh=int(0.95 * len(x.loc[a:b]))), m.loc[a:b]
        rows.append((f"{a}–{b[-2:]}", avg_corr_matrix(xx[mm > mm.quantile(0.5)]),
                     avg_corr_matrix(xx), avg_corr_matrix(xx[mm <= mm.quantile(0.10)])))
    lab = [r[0] for r in rows]
    yy = np.arange(len(rows))
    for j, (name, col, mk) in enumerate([("calm days", C_AQUA, "s"), ("all days", C_GRAY, "o"),
                                          ("stress days", C_ORANGE, "D")]):
        a3.scatter([r[j + 1] for r in rows], yy, color=col, marker=mk, s=40, label=name, zorder=3)
    for i, r in enumerate(rows):
        a3.plot([r[1], r[3]], [i, i], color=C_GRID, lw=2, zorder=1)
    a3.set_yticks(yy); a3.set_yticklabels(lab); a3.invert_yaxis()
    a3.set_xlabel("average pairwise correlation")
    a3.set_title("(c) Correlation by market state", loc="left")
    a3.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=7.5, handletextpad=0.2, columnspacing=0.8)
    return fig, pd.DataFrame(rows, columns=["Period", "Calm days", "All days", "Stress days"]).set_index("Period")


def fig_frontiers(fronts, tabs):
    """2x2: rows = samples, cols = (vol, max drawdown) vs out-of-sample stress-day correlation."""
    groups_color = {"Proposed": C_BLUE, "Proposed-2": C_VIOLET, "Original": C_ORANGE,
                    "Baseline": C_AQUA, "Benchmark": C_GRAY, "Random": C_GRAY}
    keep = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Top-k Quality", "TailDep-Cluster",
            "Corr-Cluster", "MST-Periphery", "PCA-KMeans (Vo et al.)", "EW-All (1/N)", "HRP-All"]
    short = {"B-SR-DAS (balanced)": "B-SR-DAS", "S-SR-DAS (screen+diversify)": "S-SR-DAS",
             "PCA-KMeans (Vo et al.)": "PCA-KMeans", "EW-All (1/N)": "1/N"}
    fig, axes = plt.subplots(len(fronts), 2, figsize=(10, 4.2 * len(fronts)), squeeze=False)
    for i, (label, front) in enumerate(fronts.items()):
        tab = tabs[label]
        for j, ycol in enumerate(["Vol", "MaxDD"]):
            ax = axes[i, j]
            f = front.sort_values("theta")
            yv = f[ycol].astype(float) if ycol == "Vol" else -f[ycol].astype(float)
            ax.plot(f["StressCorr_OOS"].astype(float), yv, "-o", color=C_BLUE, ms=3.5, lw=1.2, alpha=0.8)
            for _, r in f.iterrows():
                if r["theta"] in (0, 0.1, 0.5, 1.0):
                    ax.annotate(f"θ={r['theta']:g}", (float(r["StressCorr_OOS"]), float(r[ycol]) if ycol == "Vol" else -float(r[ycol])),
                                fontsize=6.5, color=C_INK2, xytext=(3, 3), textcoords="offset points")
            for n in keep:
                if n not in tab.index:
                    continue
                r = tab.loc[n]
                yy = float(r[ycol]) if ycol == "Vol" else -float(r[ycol])
                prop = r["Group"] in ("Proposed", "Proposed-2")
                col = C_ORANGE if n == "Top-k Quality" else groups_color.get(r["Group"], C_GRAY)
                ax.scatter(float(r["StressCorr_OOS"]), yy, s=90 if prop else 28,
                           marker="*" if prop else "o", color=col,
                           edgecolor="white", linewidth=0.8, zorder=4)
                ax.annotate(short.get(n, n), (float(r["StressCorr_OOS"]), yy), fontsize=7, color=C_INK,
                            xytext=(4, -8), textcoords="offset points")
            ax.set_xlabel("out-of-sample stress-day correlation of held basket (lower = better)")
            ax.set_ylabel("annualized volatility" if ycol == "Vol" else "maximum drawdown (size)")
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
            ax.set_title(f"({'abcd'[2 * i + j]}) {label}: stress correlation vs {'volatility' if ycol == 'Vol' else 'drawdown'}",
                         loc="left")
    fig.text(0.01, -0.01, "Line: SR-DAS with fixed θ (quality weight). Stars: proposed methods. "
             "Colours: blue = B-SR-DAS and the θ line, violet = S-SR-DAS, orange = Top-k Quality, aqua = other selection baselines, grey = full-universe benchmarks.",
             fontsize=7.5, color=C_INK2)
    plt.tight_layout()
    return fig


def fig_wealth_drawdown(res, names, title):
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1.6, 1]})
    for n in names:
        col, ls = SERIES_STYLE.get(n, (C_GRAY, "-"))
        w = (1 + res[n].returns).cumprod()
        lw = 2.0 if n.startswith("B-SR-DAS") else 1.3
        a1.plot(w.index, w.values, color=col, ls=ls, lw=lw, label=n)
        a2.plot(w.index, (w / w.cummax() - 1).values, color=col, ls=ls, lw=lw * 0.8)
        _label_end(a1, w.index[-1], w.values[-1], f"{n.split(' (')[0]} {w.values[-1]:.1f}×", col)
    a1.set_yscale("log"); a1.set_ylabel("growth of 1 (log scale)")
    a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    a1.yaxis.set_minor_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}" if f"{v:.1g}"[0] in "25" else ""))
    a1.set_title(f"(a) {title}: cumulative wealth, net of 10 bps costs", loc="left")
    a1.legend(loc="upper left", ncol=2)
    a2.set_ylabel("drawdown"); a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    a2.set_title("(b) Drawdowns", loc="left")
    plt.tight_layout()
    return fig


def fig_theta(thetas):
    fig, ax = plt.subplots(figsize=(10, 2.8))
    for (label, th), col in zip(thetas.items(), [C_BLUE, C_VIOLET]):
        ax.plot(th.index, th.values, color=col, lw=1.3, label=f"balanced θ, {label}")
    ax.axhline(0.5, color=C_ORANGE, ls="--", lw=1.2, label="original fixed θ = 0.5")
    ax.set_ylabel("θ"); ax.legend(loc="upper right", ncol=3)
    ax.set_title("Quality weight θ chosen by the balance rule at each rebalance", loc="left")
    plt.tight_layout()
    return fig


def fig_h1(pairs):
    """Distribution across quarters of the stress-day basket correlation difference
    (B-SR-DAS minus Top-k Quality). Negative = B-SR-DAS more diversified in stress."""
    fig, ax = plt.subplots(figsize=(7.5, 3.0))
    rng = np.random.default_rng(0)
    for i, (label, d) in enumerate(pairs.items()):
        d = d.dropna()
        y = i + rng.uniform(-0.12, 0.12, len(d))
        ax.scatter(d.values, y, s=10, color=C_BLUE, alpha=0.55, lw=0)
        ax.plot([d.median()] * 2, [i - 0.25, i + 0.25], color=C_INK, lw=2)
        ax.annotate(f"median {d.median():+.2f}; below 0 in {(d < 0).mean():.0%} of quarters",
                    (d.median(), i + 0.3), fontsize=7.5, color=C_INK, ha="center")
    ax.axvline(0, color=C_INK2, lw=1, ls=":")
    ax.set_yticks(range(len(pairs))); ax.set_yticklabels(list(pairs))
    ax.set_xlabel("stress-day basket correlation: B-SR-DAS minus Top-k Quality (per quarter)")
    ax.set_title("H1: B-SR-DAS holds a basket that is less correlated on stress days", loc="left")
    ax.set_ylim(-0.6, len(pairs) - 0.3)
    plt.tight_layout()
    return fig


def fig_robustness(rob):
    """Dot plot of Sharpe and stress correlation of B-SR-DAS vs Top-k and 1/N per configuration."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.32 * len(rob) + 1.6), sharey=True)
    y = np.arange(len(rob))
    for ax, metric in zip(axes, ["Sharpe", "StressCorr"]):
        for short, lab, col, mk in [("B-SR-DAS", "B-SR-DAS", C_BLUE, "o"), ("TopK", "Top-k Quality", C_ORANGE, "s"),
                                    ("1/N", "1/N", C_GRAY, "D")]:
            ax.scatter(rob[f"{metric} {short}"].astype(float), y, color=col, marker=mk, s=30, label=lab,
                       edgecolor="white", linewidth=0.7, zorder=3)
        ax.set_title("Sharpe ratio" if metric == "Sharpe" else "Stress-day basket correlation (lower = better)", loc="left")
    axes[0].set_yticks(y); axes[0].set_yticklabels(rob["Configuration"]); axes[0].invert_yaxis()
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    return fig


In [ ]:
import io
import os
import zipfile
import requests

FRENCH_BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"


def _download_zip_text(filename, cache_dir="data"):
    """Download a Kenneth French zip file (cached locally) and return the CSV text."""
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, filename)
    if not os.path.exists(path):
        url = FRENCH_BASE + filename
        print("Downloading", url)
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (research script)"}, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    with zipfile.ZipFile(path) as z:
        name = [n for n in z.namelist() if n.lower().endswith((".csv", ".txt"))][0]
        return z.read(name).decode("latin-1")


def parse_french_csv(text, block=0):
    """Parse a Kenneth French CSV file and return block `block` as a DataFrame.

    The files contain one or more tables (e.g. value-weighted, then equal-weighted
    returns). Each table starts with a header line beginning with ',' and continues
    with rows whose first field is a YYYYMMDD date. Values are in percent and
    missing values are coded -99.99 or -999; both are converted to NaN and
    returns are converted to decimals.
    """
    lines = text.splitlines()
    blocks, cur, header = [], None, None
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if line.startswith(",") and len(parts) > 2:
            header = parts[1:]
            cur = []
            blocks.append((header, cur))
            continue
        if cur is not None:
            if len(parts) >= 2 and parts[0].isdigit() and len(parts[0]) == 8:
                cur.append(parts[: len(header) + 1])
            elif cur:  # end of the current table
                cur = None
    header, rows = blocks[block]
    df = pd.DataFrame(rows, columns=["date"] + header)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df = df.set_index("date").apply(pd.to_numeric, errors="coerce")
    df = df.mask(df <= -99.99)
    return df / 100.0


def load_french_industries(n=49, start="1985-01-01", cache_dir="data", weighting="value"):
    """Daily returns of the Fama-French n-industry portfolios (decimal)."""
    text = _download_zip_text(f"{n}_Industry_Portfolios_daily_CSV.zip", cache_dir)
    df = parse_french_csv(text, block=0 if weighting == "value" else 1)
    return df.loc[start:]


def load_french_factors(start="1985-01-01", cache_dir="data"):
    """Daily Fama-French factors: returns DataFrame with Mkt (total market return), Mkt-RF, RF."""
    text = _download_zip_text("F-F_Research_Data_Factors_daily_CSV.zip", cache_dir)
    f = parse_french_csv(text, block=0)
    f["Mkt"] = f["Mkt-RF"] + f["RF"]
    return f.loc[start:]


In [ ]:
import os
import io
import time
import requests

MEMBERSHIP_URL = ("https://raw.githubusercontent.com/fja05680/sp500/master/"
                  "S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv")


def load_sp500_membership(cache_dir="data"):
    """Point-in-time S&P 500 membership (fja05680/sp500, based on Clenow's list + updates).

    Each row is a date on which the index changed, with the full member list on that
    date. Membership on any day = the most recent row on or before that day.
    Returns a Series: date -> list of tickers (exchange-style, e.g. 'BRK.B').
    """
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, "sp500_membership.csv")
    if not os.path.exists(path):
        print("Downloading S&P 500 membership history ...")
        r = requests.get(MEMBERSHIP_URL, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    s = df.set_index("date")["tickers"].map(lambda x: sorted(set(t.strip() for t in x.split(",") if t.strip())))
    return s.sort_index()


def yahoo_symbol(t):
    """Exchange ticker -> Yahoo symbol (share classes use '-', e.g. BRK.B -> BRK-B)."""
    return t.replace(".", "-")


def make_universe_fn(membership):
    """Return f(date) -> list of Yahoo symbols that were S&P 500 members on that date."""
    dates = membership.index

    def universe(date):
        i = dates.searchsorted(pd.Timestamp(date), side="right") - 1
        if i < 0:
            return []
        return [yahoo_symbol(t) for t in membership.iloc[i]]
    return universe


def tickers_ever(membership, start, end=None):
    """All tickers that were members at any time in [start, end]."""
    m = membership.loc[:end] if end else membership
    first = m.index.searchsorted(pd.Timestamp(start), side="right") - 1
    rows = m.iloc[max(first, 0):]
    return sorted({yahoo_symbol(t) for lst in rows for t in lst})


In [ ]:
def _yf_download(tickers, start, end):
    """Thin wrapper around yfinance (kept separate so it can be swapped or mocked)."""
    import yfinance as yf
    df = yf.download(tickers, start=start, end=end, auto_adjust=True, actions=False,
                     progress=False, threads=True, group_by="column")
    if df is None or len(df) == 0:
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        close = df["Close"]
    else:
        close = df[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    return close


def download_prices(tickers, start, end=None, cache_dir="data/prices", chunk=50, pause=1.5, retries=3):
    """Download adjusted close prices in chunks, caching each chunk to disk.

    Re-running skips chunks already on disk, so an interrupted download (rate limit,
    Colab disconnect) resumes where it stopped. Point cache_dir at Google Drive to
    keep the data between Colab sessions.
    """
    os.makedirs(cache_dir, exist_ok=True)
    tickers = sorted(set(tickers))
    frames = []
    chunks = [tickers[i:i + chunk] for i in range(0, len(tickers), chunk)]
    for ci, ch in enumerate(chunks):
        path = os.path.join(cache_dir, f"chunk_{ci:03d}_{len(tickers)}_{start}.pkl")
        if os.path.exists(path):
            frames.append(pd.read_pickle(path))
            continue
        for attempt in range(retries):
            try:
                px = _yf_download(ch, start, end)
                break
            except Exception as e:  # rate limit / network hiccup
                print(f"  chunk {ci}: attempt {attempt + 1} failed ({e}); retrying")
                time.sleep(pause * 5 * (attempt + 1))
                px = pd.DataFrame()
        px = px.reindex(columns=ch) if len(px) else pd.DataFrame(columns=ch)
        px.to_pickle(path)
        frames.append(px)
        print(f"  chunk {ci + 1}/{len(chunks)}: {px.notna().any().sum()}/{len(ch)} tickers with data")
        time.sleep(pause)
    prices = pd.concat(frames, axis=1).sort_index()
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    return prices.loc[:, ~prices.columns.duplicated()]


In [ ]:
def prices_to_returns(prices, min_share=0.5, spike=0.4, verbose=True):
    """Convert prices to clean daily simple returns.

    * returns computed per ticker without forward-filling across gaps;
    * days on which fewer than `min_share` of the tickers that are trading have
      data are dropped (holiday/feed artefacts);
    * one-day spike-and-reversal errors (|r_t| > spike followed by an opposite move
      that undoes it to within 5%) are set to NaN — a common Yahoo data error.
    Returns (returns, report) where report lists the corrections.
    """
    px = prices.sort_index()
    active = px.notna().sum(axis=1)
    keep = active >= min_share * active.rolling(20, min_periods=1).median()
    px = px[keep]
    rets = px.pct_change(fill_method=None)
    nxt = rets.shift(-1)
    spike_mask = (rets.abs() > spike) & (nxt.abs() > spike / 2) & (np.sign(rets) != np.sign(nxt)) & \
                 (((1 + rets) * (1 + nxt) - 1).abs() < 0.05)
    both = spike_mask | spike_mask.shift(1, fill_value=False)
    n_fix = int(both.values.sum())
    rets = rets.mask(both)
    extreme = rets.stack()
    extreme = extreme[extreme.abs() > 0.5]
    report = dict(days_dropped=int((~keep).sum()), spike_points_removed=n_fix,
                  remaining_moves_over_50pct=len(extreme))
    if verbose:
        print("Cleaning report:", report)
    return rets.iloc[1:], report, extreme


def coverage_audit(membership, returns, rebalance_dates, lookback, min_coverage):
    """For each rebalance date: index members, members with any Yahoo data,
    and members eligible for selection (enough history in the look-back)."""
    uni = make_universe_fn(membership)
    rows = []
    idx = returns.index
    for d in rebalance_dates:
        t = idx.searchsorted(d, side="right") - 1
        mem = uni(d)
        have = [m for m in mem if m in returns.columns]
        win = returns.iloc[max(0, t - lookback + 1): t + 1].reindex(columns=have)
        cov = win.notna().mean()
        elig = int(((cov >= min_coverage) & win.iloc[-1].notna()).sum()) if len(have) else 0
        rows.append(dict(date=d, members=len(mem), with_yahoo_data=int((cov > 0).sum()),
                         eligible=elig))
    df = pd.DataFrame(rows).set_index("date")
    df["coverage_%"] = 100 * df["eligible"] / df["members"]
    return df


In [ ]:
pub_style()
print('library loaded')

## 3. Data, Table 1 and Figure 2

In [ ]:
ind = load_french_industries(49, start="1963-07-01", cache_dir=DATA_DIR)
ff = load_french_factors(start="1963-07-01", cache_dir=DATA_DIR)
common = ind.index.intersection(ff.index)
ind, ff = ind.loc[common], ff.loc[common]
MKT, RF = ff["Mkt"], ff["RF"]
H0, H1_ = SPEC["holdout_period"]
print(f"49 industries: {ind.index[0].date()} to {ind.index[-1].date()}")

t1 = pd.DataFrame([descriptive_block(ind, MKT, H0, H1_, "Industries, holdout"),
                   descriptive_block(ind, MKT, "1990-01-01", None, "Industries, replication")]).set_index("Sample")
export_table(t1.T, "T1_descriptive_industries", "Descriptive statistics of the industry samples")
display(t1.T)

In [ ]:
fig, motiv = fig_motivation(ind, MKT)
save_fig(fig, "Fig2_correlation_in_stress")
export_table(motiv, "T1b_correlation_by_state", "Average pairwise industry correlation by market state")
display(motiv.round(3))

## 4. Main results: holdout 1970–1989 (headline) and replication 1990–latest

In [ ]:
from dataclasses import replace
CFG_BASE = Config(k=SPEC["k"], lookback=SPEC["lookback_days"], rebalance_every=SPEC["rebalance_days"],
                  stress_q=SPEC["stress_q"], sub_len=SPEC["sub_len"], sub_step=SPEC["sub_step"],
                  agg_q=SPEC["agg_q"], kernel_a=SPEC["kernel_a"], cost_bps=SPEC["cost_bps"],
                  theta=0.5, n_random=N_RANDOM)
SAMPLES = {"Holdout 1970-1989": replace(CFG_BASE, test_start=H0, test_end=H1_),
           "Replication 1990+": replace(CFG_BASE, test_start="1990-01-01", test_end=None)}
RES, TABS, THETAS, HYPS = {}, {}, {}, {}
for label, cfg in SAMPLES.items():
    print(label)
    THETA_LOG.clear()
    RES[label] = run_backtest(ind, MKT, v3_strategies(cfg), cfg)
    THETAS[label] = pd.Series(THETA_LOG).sort_index()
    TABS[label] = summary_table(RES[label], RF, market=MKT)
    HYPS[label] = evaluate_hypotheses(RES[label], "B-SR-DAS (balanced)", RF, B=N_BOOT)

In [ ]:
KEY = ["Group", "CAGR", "Vol", "Sharpe", "MaxDD", "Calmar", "Turnover/yr", "StressCorr_OOS", "StressBeta"]
for label, tag in [("Holdout 1970-1989", "T2_main_holdout"), ("Replication 1990+", "T2b_main_replication")]:
    export_table(TABS[label][KEY], tag, f"Out-of-sample performance, 49 industries, {label}")
    print(label); display(format_table(TABS[label][KEY]))

In [ ]:
fmt = lambda h: h["Verdict"] + " (" + h["Statistic"] + "; " + h["Detail"] + ")"
t3 = pd.DataFrame({"Hypothesis": HYPS["Holdout 1970-1989"]["Description"],
                   "Holdout 1970-1989": fmt(HYPS["Holdout 1970-1989"]),
                   "Replication 1990+": fmt(HYPS["Replication 1990+"])})
export_table(t3, "T3_hypotheses", "Pre-registered hypotheses: verdicts")
display(t3)

names_ci = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Original SR-DAS (theta=0.5)",
            "Original SR-DAS (theta=0)", "TailDep-Cluster", "Corr-Cluster", "MST-Periphery",
            "PCA-KMeans (Vo et al.)", "Top-k Quality", "EW-All (1/N)", "HRP-All"]
t4 = pd.concat({lab: sharpe_ci_table(RES[lab], names_ci, RF, B=N_BOOT) for lab in RES}, axis=1)
export_table(t4, "T4_sharpe_ci", "Annualized Sharpe ratios with block-bootstrap 95% confidence intervals")
display(t4.round(3))
sh = pd.concat({lab: sharpe_tests_holm(RES[lab], "B-SR-DAS (balanced)", RF, B=N_BOOT) for lab in RES}, axis=1)
export_table(sh, "T4b_sharpe_tests_holm", "Sharpe differences vs B-SR-DAS with Holm-adjusted p-values")

In [ ]:
CRISES = {"Holdout 1970-1989": {"1973-74 bear": ("1973-01-11", "1974-10-03"), "1980 Q1": ("1980-02-13", "1980-03-27"),
                                "1981-82 recession": ("1980-11-28", "1982-08-12"), "1987 crash": ("1987-08-25", "1987-12-04")},
          "Replication 1990+": {"Gulf war 1990": ("1990-07-16", "1990-10-11"), "LTCM 1998": ("1998-07-17", "1998-10-08"),
                                "Dot-com 2000-02": ("2000-03-24", "2002-10-09"), "GFC 2007-09": ("2007-10-09", "2009-03-09"),
                                "Euro debt 2011": ("2011-04-29", "2011-10-03"), "China/oil 2015-16": ("2015-05-21", "2016-02-11"),
                                "COVID 2020": ("2020-02-19", "2020-03-23"), "Rates 2022": ("2022-01-03", "2022-10-12"),
                                "Tariffs 2025": ("2025-02-19", "2025-04-08")}}
crisis_rows = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Original SR-DAS (theta=0.5)",
               "Original SR-DAS (theta=0)", "TailDep-Cluster", "Corr-Cluster", "Top-k Quality", "EW-All (1/N)",
               "HRP-All", "Market"]
for lab, tag in [("Holdout 1970-1989", "T5_crisis_holdout"), ("Replication 1990+", "T5b_crisis_replication")]:
    dd, _ = crisis_table(RES[lab], CRISES[lab], MKT)
    dd = dd.reindex([r for r in crisis_rows if r in dd.index])
    export_table(dd, tag, f"Maximum drawdown within crisis windows, {lab}")
    print(lab); display(dd.style.format("{:.1%}"))

### Figures 1, 3–7

In [ ]:
save_fig(fig_framework(), "Fig1_framework")
show = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Top-k Quality", "TailDep-Cluster", "EW-All (1/N)"]
save_fig(fig_wealth_drawdown(RES["Holdout 1970-1989"], show, "Holdout 1970-1989"), "Fig3_wealth_holdout")
save_fig(fig_wealth_drawdown(RES["Replication 1990+"], show, "Replication 1990-latest"), "Fig4_wealth_replication")
pairs = {lab: (RES[lab]["B-SR-DAS (balanced)"].diag["stress_corr_oos"] - RES[lab]["Top-k Quality"].diag["stress_corr_oos"])
         for lab in RES}
save_fig(fig_h1(pairs), "Fig5_H1_stress_corr_difference")
save_fig(fig_theta(THETAS), "Fig6_balanced_theta")
th = pd.DataFrame({lab: s.describe()[["mean", "50%", "min", "max"]] for lab, s in THETAS.items()}).T
export_table(th, "T3b_balanced_theta", "Balanced theta chosen by the rule")

In [ ]:
THETA_GRID = [0.0, 0.025, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
FRONTS = {}
for lab, cfg in SAMPLES.items():
    print("frontier:", lab)
    FRONTS[lab] = run_frontier(ind, MKT, cfg, THETA_GRID, RF)
    export_table(FRONTS[lab][["theta", "CAGR", "Vol", "Sharpe", "MaxDD", "StressCorr_OOS", "StressBeta"]],
                 "T2c_frontier_" + ("holdout" if "Holdout" in lab else "replication"), f"Fixed-theta frontier, {lab}")
save_fig(fig_frontiers(FRONTS, TABS), "Fig7_frontiers")

## 5. Robustness of B-SR-DAS (Table 6, Figure 8)

Secondary analyses of the frozen method: transaction costs (re-priced exactly from gross returns and turnover), portfolio size, min-variance weights, and the 12-industry data. For each configuration: Sharpe, volatility and stress-day correlation of B-SR-DAS, Top-k Quality and 1/N, plus whether H1–H4 hold (`yes`/`no`).

In [ ]:
ROB_NAMES = ["B-SR-DAS (balanced)", "Top-k Quality", "EW-All (1/N)", "Original SR-DAS (theta=0)", "TailDep-Cluster"]
def subset(cfg, weights="ew"):
    return [s for s in v3_strategies(cfg, include_random=False, weights=weights) if s.name in ROB_NAMES]

rob = []
if RUN_ROBUSTNESS:
    ind12 = load_french_industries(12, start="1963-07-01", cache_dir=DATA_DIR)
    ind12 = ind12.loc[ind12.index.intersection(MKT.index)]
    for lab, cfg in SAMPLES.items():
        short = "holdout" if "Holdout" in lab else "replication"
        base = {n: RES[lab][n] for n in ROB_NAMES}
        for bps in [0, 10, 25, 50]:
            r = base if bps == SPEC["cost_bps"] else apply_costs(base, bps)
            rob.append(robustness_rows(r, f"{short}: costs {bps} bps" + (" (base)" if bps == 10 else ""), RF, N_BOOT_ROBUST))
        for k in [5, 15]:
            c = replace(cfg, k=k)
            rob.append(robustness_rows(run_backtest(ind, MKT, subset(c), c, verbose=False), f"{short}: k = {k}", RF, N_BOOT_ROBUST))
        rob.append(robustness_rows(run_backtest(ind, MKT, subset(cfg, "minvar"), cfg, verbose=False),
                                   f"{short}: min-variance weights", RF, N_BOOT_ROBUST))
        c12 = replace(cfg, k=4)
        rob.append(robustness_rows(run_backtest(ind12, MKT, subset(c12), c12, verbose=False),
                                   f"{short}: 12 industries, k = 4", RF, N_BOOT_ROBUST))
        print("done:", lab)
    t6 = pd.DataFrame(rob).set_index("Configuration")
    export_table(t6, "T6_robustness", "Robustness of B-SR-DAS")
    display(t6.round(3))
    save_fig(fig_robustness(t6.reset_index()), "Fig8_robustness")

## 6. S&P 500 replication (Tables 7–9, Figures 9–10)

Point-in-time S&P 500 membership (fja05680/sp500) and Yahoo Finance prices, exactly as in notebook 2. This sample was seen during development, so it is **replication evidence**, not confirmation. Coverage of past index members is incomplete (Yahoo lacks many delisted firms); the coverage table and the survivorship check quantify this.

In [ ]:
if RUN_SP500:
    membership = load_sp500_membership(SP500_DATA_DIR)
    universe = make_universe_fn(membership)
    all_tickers = tickers_ever(membership, "2000-01-01", None)
    prices = download_prices(all_tickers + ["^GSPC"], "2000-01-01", None,
                             cache_dir=os.path.join(SP500_DATA_DIR, "prices"))
    rets_all, clean_report, extreme = prices_to_returns(prices)
    MKT_SP = rets_all.pop("^GSPC")
    rets = rets_all
    RF_SP = load_french_factors(start="2000-01-01", cache_dir=DATA_DIR)["RF"].reindex(rets.index).ffill().fillna(0.0)
    ok = MKT_SP.notna()
    rets, MKT_SP, RF_SP = rets[ok], MKT_SP[ok], RF_SP[ok]
    n_missing = sum(1 for t in all_tickers if t not in prices.columns or prices[t].notna().sum() == 0)
    print(f"tickers ever in index since 2000: {len(all_tickers)}; no Yahoo data: {n_missing} ({n_missing/len(all_tickers):.0%})")

In [ ]:
if RUN_SP500:
    CFG_SP = replace(CFG_BASE, test_start="2005-01-01", test_end=None, n_random=N_RANDOM_SP500)
    t0 = max(rets.index.searchsorted(pd.Timestamp("2005-01-01")), CFG_SP.lookback)
    audit = coverage_audit(membership, rets, rets.index[t0 - 1::CFG_SP.rebalance_every], CFG_SP.lookback, CFG_SP.min_coverage)
    t9 = audit.groupby(audit.index.year).mean().round(1)
    export_table(t9, "T9_sp500_coverage", "S&P 500 members usable with Yahoo data (yearly averages)", "{:.1f}")
    display(t9)
    print("running S&P 500 backtest ...")
    THETA_LOG.clear()
    RES_SP = run_backtest(rets, MKT_SP, v3_strategies(CFG_SP), CFG_SP, universe_fn=universe)
    TAB_SP = summary_table(RES_SP, RF_SP, market=MKT_SP)
    export_table(TAB_SP[KEY], "T7_sp500_main", "Out-of-sample performance, S&P 500 stocks 2005-latest")
    display(format_table(TAB_SP[KEY]))
    HYP_SP = evaluate_hypotheses(RES_SP, "B-SR-DAS (balanced)", RF_SP, B=N_BOOT)
    export_table(HYP_SP, "T7b_sp500_hypotheses", "Hypotheses on S&P 500 stocks (replication)")
    display(HYP_SP)
    export_table(sharpe_tests_holm(RES_SP, "B-SR-DAS (balanced)", RF_SP, B=N_BOOT), "T7c_sp500_sharpe_tests_holm",
                 "Sharpe differences vs B-SR-DAS, S&P 500")

In [ ]:
if RUN_SP500:
    key = ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Abl: Stress-Cluster (rule)", "TailDep-Cluster",
           "Corr-Cluster", "Top-k Quality", "EW-All (1/N)"]
    today = set(universe(rets.index[-1]))
    res_surv = run_backtest(rets, MKT_SP, [s for s in v3_strategies(CFG_SP, include_random=False) if s.name in key],
                            CFG_SP, universe_fn=lambda d: list(today), verbose=False)
    a = summary_table({n: RES_SP[n] for n in key}, RF_SP, market=MKT_SP)
    b = summary_table(res_surv, RF_SP, market=MKT_SP)
    t8 = pd.DataFrame({"Sharpe point-in-time": a["Sharpe"], "Sharpe survivors-only": b["Sharpe"],
                       "StressCorr point-in-time": a["StressCorr_OOS"], "StressCorr survivors-only": b["StressCorr_OOS"]}).astype(float)
    t8["Sharpe rank PIT"] = t8["Sharpe point-in-time"].rank(ascending=False)
    t8["Sharpe rank survivors"] = t8["Sharpe survivors-only"].rank(ascending=False)
    export_table(t8, "T8_sp500_survivorship", "Survivorship check: point-in-time vs today's members")
    display(t8.round(3))
    save_fig(fig_wealth_drawdown(RES_SP, ["B-SR-DAS (balanced)", "S-SR-DAS (screen+diversify)", "Top-k Quality",
                                          "TailDep-Cluster", "EW-All (1/N)"], "S&P 500 stocks 2005-latest"), "Fig9_wealth_sp500")
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.plot(audit.index, audit["members"], color=C_GRAY, lw=1.2, label="index members")
    ax.plot(audit.index, audit["eligible"], color=C_BLUE, lw=1.6, label="usable (Yahoo data, ≥95% history)")
    ax.set_ylim(0, 550); ax.legend(loc="lower right"); ax.set_ylabel("stocks")
    ax.set_title("Coverage of point-in-time S&P 500 members", loc="left")
    plt.tight_layout(); save_fig(fig, "Fig10_sp500_coverage")

## 7. Export everything

In [ ]:
write_workbook(os.path.join(TAB_DIR, "all_tables.xlsx"))
with open(os.path.join(OUT_ROOT, "specification.json"), "w") as f:
    f.write(SPEC_JSON + "\n\nfingerprint: " + SPEC_HASH + "\n")
import shutil, platform, sklearn, scipy, numpy
with open(os.path.join(OUT_ROOT, "environment.txt"), "w") as f:
    f.write(f"python {platform.python_version()}\nnumpy {numpy.__version__}\npandas {pd.__version__}\n"
            f"scipy {scipy.__version__}\nscikit-learn {sklearn.__version__}\nnetworkx {nx.__version__}\n")
shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)
print("tables:", sorted(os.listdir(TAB_DIR)))
print("figures:", sorted(os.listdir(FIG_DIR)))
try:
    from google.colab import files
    files.download(OUT_ROOT + ".zip")
except Exception:
    print("archive:", OUT_ROOT + ".zip")